# 🧪 LAB: Fine-Tuning LLMs for Domain-Specific Projects

> **Level:** Intermediate → Advanced  
> **Duration:** ~4–6 hours per lab  
> **Prerequisites:** Python, Hugging Face Transformers, GPU or Google Colab  

---

## 📚 Labs Covered
| Lab | Domain | Model | Technique |
|-----|--------|-------|-----------|
| Lab 1 | 🏥 Medical Q&A | Mistral / TinyLlama | QLoRA + SFTTrainer |
| Lab 2 | ⚖️ Legal Summarization | FLAN-T5 | Seq2Seq Fine-Tuning |
| Lab 3 | 📈 Financial Sentiment | FinBERT | Classification |
| Lab 4 | 🛒 Customer Support Bot | TinyLlama | Chat Template + QLoRA |
| Lab 5 | 💻 Code Generation | CodeLlama | Instruction Tuning |

## ⚙️ Environment Setup (Run Once)
Install all required libraries before starting any lab.

In [ ]:
# Install all required libraries
!pip install -q transformers datasets peft trl accelerate bitsandbytes \
               sentencepiece huggingface_hub rouge_score bert_score gradio

In [ ]:
# Check GPU availability
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU — use Google Colab!'}")

---
# 🔬 LAB 1 — Medical Q&A Bot (Healthcare Domain)

**Objective:** Fine-tune a small LLM on medical Q&A pairs using QLoRA.  
**Dataset:** MedQuAD — Medical Question & Answer pairs  
**Model:** TinyLlama-1.1B (low GPU) or Mistral-7B (high GPU)  

### ✅ Lab 1 Checklist
- [ ] Dataset loaded and formatted
- [ ] QLoRA config applied
- [ ] Model trained
- [ ] Inference tested with 5 medical questions

In [ ]:
# --- LAB 1: Step 1 — Load Dataset ---
from datasets import load_dataset

dataset = load_dataset("keivalya/MedQuad-MedicalQnADataset", split="train")
print(f"Total samples: {len(dataset)}")
print(dataset[0])

In [ ]:
# --- LAB 1: Step 2 — Format into Instruction Template ---
def format_medical(example):
    return {
        "text": (
            "### Instruction:\n"
            "You are a medical assistant. Answer the following question accurately.\n\n"
            f"### Question:\n{example['Question']}\n\n"
            f"### Answer:\n{example['Answer']}"
        )
    }

dataset = dataset.map(format_medical)
print(dataset[0]["text"][:400])

In [ ]:
# --- LAB 1: Step 3 — Load Model with QLoRA (4-bit quantization) ---
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # Change to mistralai/Mistral-7B-v0.1 for better quality

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)
print("✅ Model loaded successfully!")

In [ ]:
# --- LAB 1: Step 4 — Apply LoRA Adapters ---
lora_config = LoraConfig(
    r=16,                        # LoRA rank — higher = more params
    lora_alpha=32,               # scaling factor
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expected: ~1-2% of total params are trainable — very memory efficient!

In [ ]:
# --- LAB 1: Step 5 — Train with SFTTrainer ---
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./outputs/medical-llm",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=50,
    save_strategy="epoch",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=512,
    tokenizer=tokenizer,
    args=training_args,
)

trainer.train()
print("✅ Training complete!")

In [ ]:
# --- LAB 1: Step 6 — Save & Run Inference ---
model.save_pretrained("./outputs/medical-llm-adapter")
tokenizer.save_pretrained("./outputs/medical-llm-adapter")

def ask_medical(question):
    prompt = (
        "### Instruction:\nAnswer this medical question.\n\n"
        f"### Question:\n{question}\n\n### Answer:\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=200, do_sample=False)
    return tokenizer.decode(output[0], skip_special_tokens=True).split("### Answer:")[-1].strip()

# Test inference
questions = [
    "What are the symptoms of diabetes?",
    "How is hypertension diagnosed?",
    "What causes anemia?"
]
for q in questions:
    print(f"Q: {q}")
    print(f"A: {ask_medical(q)}\n{'-'*60}")

---
# 🔬 LAB 2 — Legal Document Summarizer (Legal Domain)

**Objective:** Fine-tune FLAN-T5 to summarize complex legal text into plain English.  
**Dataset:** legal_summarization from Hugging Face  
**Model:** google/flan-t5-base  

### ✅ Lab 2 Checklist
- [ ] Dataset loaded and tokenized
- [ ] FLAN-T5 fine-tuned
- [ ] ROUGE score evaluated
- [ ] 3 legal docs summarized

In [ ]:
# --- LAB 2: Step 1 — Load Dataset ---
from datasets import load_dataset

legal_ds = load_dataset("joelniklaus/legal_summarization", split="train[:2000]")
print(f"Samples: {len(legal_ds)}")
print("Keys:", legal_ds.column_names)

In [ ]:
# --- LAB 2: Step 2 — Load FLAN-T5 ---
from transformers import T5ForConditionalGeneration, T5Tokenizer

t5_tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-base")
t5_model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-base")
print("✅ FLAN-T5 loaded!")

In [ ]:
# --- LAB 2: Step 3 — Tokenize Dataset ---
def preprocess_legal(example):
    prefix = "summarize in plain English: "
    inputs = t5_tokenizer(
        prefix + example["document"][:1024],
        max_length=512, truncation=True, padding="max_length"
    )
    labels = t5_tokenizer(
        example["summary"],
        max_length=128, truncation=True, padding="max_length"
    )
    inputs["labels"] = labels["input_ids"]
    return inputs

tokenized_legal = legal_ds.map(preprocess_legal, batched=True, remove_columns=legal_ds.column_names)
tokenized_legal.set_format("torch")
print("✅ Dataset tokenized!")

In [ ]:
# --- LAB 2: Step 4 — Fine-Tune ---
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq

t5_args = Seq2SeqTrainingArguments(
    output_dir="./outputs/legal-summarizer",
    num_train_epochs=4,
    per_device_train_batch_size=8,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    save_total_limit=2,
    logging_steps=50,
)

t5_trainer = Seq2SeqTrainer(
    model=t5_model,
    args=t5_args,
    train_dataset=tokenized_legal,
    data_collator=DataCollatorForSeq2Seq(t5_tokenizer, model=t5_model),
)
t5_trainer.train()
print("✅ Training complete!")

In [ ]:
# --- LAB 2: Step 5 — Evaluate with ROUGE ---
from rouge_score import rouge_scorer

def summarize_legal(text):
    input_ids = t5_tokenizer(
        "summarize in plain English: " + text[:1024],
        return_tensors="pt", max_length=512, truncation=True
    ).input_ids
    out = t5_model.generate(input_ids, max_new_tokens=128)
    return t5_tokenizer.decode(out[0], skip_special_tokens=True)

scorer = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True)

# Test on first 3 examples
for i in range(3):
    sample = legal_ds[i]
    pred = summarize_legal(sample["document"])
    scores = scorer.score(pred, sample["summary"])
    print(f"Sample {i+1} | ROUGE-1: {scores['rouge1'].fmeasure:.3f} | ROUGE-L: {scores['rougeL'].fmeasure:.3f}")
    print(f"Predicted: {pred[:200]}\n")

---
# 🔬 LAB 3 — Financial Sentiment Analyzer (Finance Domain)

**Objective:** Fine-tune FinBERT to classify financial news as Positive / Neutral / Negative.  
**Dataset:** financial_phrasebank  
**Model:** yiyanghkust/finbert-tone  

### ✅ Lab 3 Checklist
- [ ] FinBERT loaded and fine-tuned
- [ ] Accuracy > 85% on validation
- [ ] Streamlit app created

In [ ]:
# --- LAB 3: Step 1 — Load Dataset ---
from datasets import load_dataset

fin_ds = load_dataset("financial_phrasebank", "sentences_75agree", split="train")
fin_ds = fin_ds.train_test_split(test_size=0.2, seed=42)
print(f"Train: {len(fin_ds['train'])} | Test: {len(fin_ds['test'])}")
print("Label: 0=Negative, 1=Neutral, 2=Positive")
print(fin_ds["train"][0])

In [ ]:
# --- LAB 3: Step 2 — Tokenize ---
from transformers import BertTokenizer, BertForSequenceClassification

bert_tokenizer = BertTokenizer.from_pretrained("yiyanghkust/finbert-tone")
bert_model = BertForSequenceClassification.from_pretrained("yiyanghkust/finbert-tone", num_labels=3)

def tokenize_fin(example):
    return bert_tokenizer(example["sentence"], truncation=True, padding="max_length", max_length=128)

tokenized_fin = fin_ds.map(tokenize_fin, batched=True)
tokenized_fin = tokenized_fin.rename_column("label", "labels")
tokenized_fin.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
print("✅ Tokenization complete!")

In [ ]:
# --- LAB 3: Step 3 — Fine-Tune FinBERT ---
from transformers import Trainer, TrainingArguments
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted")
    }

bert_args = TrainingArguments(
    output_dir="./outputs/financial-sentiment",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    fp16=torch.cuda.is_available(),
)

bert_trainer = Trainer(
    model=bert_model,
    args=bert_args,
    train_dataset=tokenized_fin["train"],
    eval_dataset=tokenized_fin["test"],
    compute_metrics=compute_metrics,
)
bert_trainer.train()
results = bert_trainer.evaluate()
print(f"\n✅ Final Accuracy: {results['eval_accuracy']:.4f} | F1: {results['eval_f1']:.4f}")

In [ ]:
# --- LAB 3: Step 4 — Run Inference ---
from transformers import pipeline

label_map = {0: "Negative 📉", 1: "Neutral ➡️", 2: "Positive 📈"}

clf_pipe = pipeline("text-classification", model=bert_model, tokenizer=bert_tokenizer)

headlines = [
    "The company reported record profits this quarter.",
    "Stock prices fell sharply after the earnings report.",
    "The board announced no changes to the dividend policy."
]

for h in headlines:
    result = clf_pipe(h)[0]
    label_idx = int(result["label"].split("_")[-1])
    print(f"📰 {h}")
    print(f"   Sentiment: {label_map[label_idx]} (score: {result['score']:.2f})\n")

---
# 🔬 LAB 4 — Customer Support Chatbot (E-Commerce Domain)

**Objective:** Fine-tune TinyLlama on customer support conversations.  
**Dataset:** Bitext customer support dataset  
**Model:** TinyLlama-1.1B-Chat  

### ✅ Lab 4 Checklist
- [ ] Dataset formatted as chat template
- [ ] TinyLlama fine-tuned
- [ ] 10 test queries answered
- [ ] Gradio demo launched

In [ ]:
# --- LAB 4: Step 1 — Load & Format Dataset ---
from datasets import load_dataset

support_ds = load_dataset(
    "bitext/Bitext-customer-support-llm-chatbot-training-dataset", split="train[:5000]"
)

def format_chat(example):
    return {
        "text": f"<s>[INST] {example['instruction']} [/INST] {example['response']} </s>"
    }

support_ds = support_ds.map(format_chat)
print(f"✅ {len(support_ds)} samples formatted")
print(support_ds[0]["text"][:300])

In [ ]:
# --- LAB 4: Step 2 — Fine-Tune TinyLlama (same QLoRA pattern as Lab 1) ---
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
import torch

SUPPORT_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                          bnb_4bit_compute_dtype=torch.float16)

sup_tok = AutoTokenizer.from_pretrained(SUPPORT_MODEL)
sup_tok.pad_token = sup_tok.eos_token
sup_model = AutoModelForCausalLM.from_pretrained(SUPPORT_MODEL, quantization_config=bnb, device_map="auto")

lora = LoraConfig(r=8, lora_alpha=16, target_modules=["q_proj","v_proj"],
                  lora_dropout=0.05, bias="none", task_type="CAUSAL_LM")
sup_model = prepare_model_for_kbit_training(sup_model)
sup_model = get_peft_model(sup_model, lora)

sup_args = TrainingArguments(
    output_dir="./outputs/support-chatbot",
    num_train_epochs=2,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=50,
)

SFTTrainer(
    model=sup_model, train_dataset=support_ds,
    dataset_text_field="text", max_seq_length=512,
    tokenizer=sup_tok, args=sup_args,
).train()
print("✅ Support chatbot trained!")

In [ ]:
# --- LAB 4: Step 3 — Launch Gradio Demo ---
import gradio as gr

def support_chat(user_input):
    prompt = f"<s>[INST] {user_input} [/INST]"
    inputs = sup_tok(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        output = sup_model.generate(**inputs, max_new_tokens=150, do_sample=True, temperature=0.7)
    response = sup_tok.decode(output[0], skip_special_tokens=True)
    return response.split("[/INST]")[-1].strip()

gr.Interface(
    fn=support_chat,
    inputs=gr.Textbox(label="Your Question", placeholder="How do I return my order?"),
    outputs=gr.Textbox(label="Support Agent Response"),
    title="🛒 AI Customer Support Chatbot",
    description="Fine-tuned TinyLlama on customer support data"
).launch(share=True)

---
# 🔬 LAB 5 — Code Generation Assistant (Software Domain)

**Objective:** Fine-tune a model to write Python code from natural language instructions.  
**Dataset:** python_code_instructions_18k_alpaca  
**Model:** bigcode/starcoderbase-1b (lightweight) or codellama/CodeLlama-7b-hf  

### ✅ Lab 5 Checklist
- [ ] Code dataset formatted with instruction template
- [ ] StarCoder / CodeLlama fine-tuned
- [ ] 5 Python functions generated and tested
- [ ] Compared base vs fine-tuned output

In [ ]:
# --- LAB 5: Step 1 — Load Dataset ---
from datasets import load_dataset

code_ds = load_dataset("iamtarun/python_code_instructions_18k_alpaca", split="train[:5000]")

def format_code(example):
    return {
        "text": (
            "### Instruction:\nWrite Python code to solve the following task.\n\n"
            f"### Task:\n{example['instruction']}\n\n"
            f"### Python Code:\n{example['output']}"
        )
    }

code_ds = code_ds.map(format_code)
print(f"✅ {len(code_ds)} samples formatted")
print(code_ds[0]["text"][:400])

In [ ]:
# --- LAB 5: Step 2 — Fine-Tune StarCoder with QLoRA ---
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
import torch

CODE_MODEL = "bigcode/starcoderbase-1b"  # Lightweight; use codellama/CodeLlama-7b-hf for best results

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                          bnb_4bit_compute_dtype=torch.float16)
code_tok = AutoTokenizer.from_pretrained(CODE_MODEL)
code_tok.pad_token = code_tok.eos_token

code_model = AutoModelForCausalLM.from_pretrained(CODE_MODEL, quantization_config=bnb, device_map="auto")

lora = LoraConfig(r=16, lora_alpha=32, target_modules=["c_attn", "c_proj"],
                  lora_dropout=0.05, bias="none", task_type="CAUSAL_LM")
code_model = prepare_model_for_kbit_training(code_model)
code_model = get_peft_model(code_model, lora)
code_model.print_trainable_parameters()

SFTTrainer(
    model=code_model, train_dataset=code_ds,
    dataset_text_field="text", max_seq_length=512,
    tokenizer=code_tok,
    args=TrainingArguments(
        output_dir="./outputs/code-assistant",
        num_train_epochs=3, per_device_train_batch_size=4,
        gradient_accumulation_steps=4, learning_rate=2e-4,
        fp16=True, logging_steps=50,
    )
).train()
print("✅ Code assistant trained!")

In [ ]:
# --- LAB 5: Step 3 — Generate & Test Code ---
def generate_code(task_description):
    prompt = (
        "### Instruction:\nWrite Python code to solve the following task.\n\n"
        f"### Task:\n{task_description}\n\n### Python Code:\n"
    )
    inputs = code_tok(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        output = code_model.generate(**inputs, max_new_tokens=200, do_sample=False)
    return code_tok.decode(output[0], skip_special_tokens=True).split("### Python Code:")[-1].strip()

tasks = [
    "Write a function to find the factorial of a number.",
    "Write a function to check if a string is a palindrome.",
    "Write a function to sort a list of dictionaries by a given key.",
]

for t in tasks:
    print(f"📝 Task: {t}")
    code = generate_code(t)
    print(f"```python\n{code}\n```\n{'='*60}")

In [ ]:
# --- LAB 5: Step 4 — Syntax Validation ---
import ast

def check_syntax(code_str):
    try:
        ast.parse(code_str)
        return "✅ Syntax OK"
    except SyntaxError as e:
        return f"❌ Syntax Error: {e}"

# Test on generated code
test_code = generate_code("Write a function to reverse a string.")
print(test_code)
print(check_syntax(test_code))

---
# 📊 Evaluation & Benchmarking

Run this section after completing any lab to evaluate model quality.

In [ ]:
# BERTScore — for open-ended generation quality
from bert_score import score as bert_score_fn

predictions = ["The patient shows elevated blood sugar levels indicating diabetes."]
references  = ["High blood glucose is a primary indicator of diabetes mellitus."]

P, R, F1 = bert_score_fn(predictions, references, lang="en", verbose=True)
print(f"BERTScore — Precision: {P.mean():.4f} | Recall: {R.mean():.4f} | F1: {F1.mean():.4f}")

In [ ]:
# ROUGE Score — for summarization quality
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

pred = "The company reported strong earnings driven by increased sales."
ref  = "Strong sales drove the company's impressive earnings this quarter."

scores = scorer.score(pred, ref)
for k, v in scores.items():
    print(f"{k}: Precision={v.precision:.3f} | Recall={v.recall:.3f} | F1={v.fmeasure:.3f}")

---
# 🚀 Deployment Options

After fine-tuning, deploy your model using one of these options.

In [ ]:
# Option A: Push adapter to Hugging Face Hub
# !huggingface-cli login

# model.push_to_hub("your-username/medical-llm-v1")
# tokenizer.push_to_hub("your-username/medical-llm-v1")
print("Uncomment and run after huggingface-cli login")

In [ ]:
# Option B: Save FastAPI app to file
api_code = '''
from fastapi import FastAPI
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

app = FastAPI()

base_model = AutoModelForCausalLM.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0", device_map="auto")
model = PeftModel.from_pretrained(base_model, "./outputs/medical-llm-adapter")
tokenizer = AutoTokenizer.from_pretrained("./outputs/medical-llm-adapter")

@app.post("/ask")
def ask(question: str):
    prompt = f"### Instruction:\\nAnswer this question.\\n\\n### Question:\\n{question}\\n\\n### Answer:\\n"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    output = model.generate(**inputs, max_new_tokens=200)
    return {"answer": tokenizer.decode(output[0], skip_special_tokens=True).split("### Answer:")[-1].strip()}
'''

with open("./outputs/api.py", "w") as f:
    f.write(api_code)

print("✅ api.py saved! Run with: uvicorn outputs.api:app --reload")

---
# 📌 Quick Reference — LoRA Hyperparameter Guide

| Parameter | Recommended | Effect |
|-----------|-------------|--------|
| `r` (rank) | 8–64 | Higher = more parameters = better quality |
| `lora_alpha` | 2× r | Scaling factor |
| `lora_dropout` | 0.05 | Regularization |
| `learning_rate` | 1e-4 to 3e-4 | Lower = more stable |
| `epochs` | 2–5 | Avoid overfitting |
| `batch_size` | 4–16 | Larger = faster & more stable |

---

# 🎓 Learning Resources

| Resource | Link |
|----------|------|
| Hugging Face PEFT Docs | https://huggingface.co/docs/peft |
| TRL SFTTrainer | https://huggingface.co/docs/trl |
| QLoRA Paper | https://arxiv.org/abs/2305.14314 |
| DeepLearning.AI Course | https://www.deeplearning.ai/short-courses/finetuning-large-language-models/ |

> 💡 **Pro Tip:** Always compare fine-tuned vs base model on 20 test examples before shipping!